In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip -q install xgboost lightgbm joblib

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
from typing import Dict, List

import json
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

warnings.filterwarnings(
    "ignore"
)

print("=" * 90)
print("IMPORTS COMPLETE")
print("=" * 90)

print(
    "Pandas :",
    pd.__version__
)

print(
    "NumPy  :",
    np.__version__
)

print(
    "XGBoost:",
    "available"
    if XGBClassifier is not None
    else "NOT AVAILABLE"
)

print(
    "LightGBM:",
    "available"
    if LGBMClassifier is not None
    else "NOT AVAILABLE"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Razorpay_AI"
)

TRAIN_PATH = (
    PROJECT_ROOT /
    "ml_data/matching/train.csv"
)

VALIDATION_PATH = (
    PROJECT_ROOT /
    "ml_data/matching/validation.csv"
)

TEST_PATH = (
    PROJECT_ROOT /
    "ml_data/matching/test.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "ml/models/reconciliation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert TRAIN_PATH.exists(), (
    f"Train dataset not found:\n{TRAIN_PATH}"
)

assert VALIDATION_PATH.exists(), (
    f"Validation dataset not found:\n{VALIDATION_PATH}"
)

assert TEST_PATH.exists(), (
    f"Test dataset not found:\n{TEST_PATH}"
)


print("\n" + "=" * 90)
print("PATH CONFIGURATION")
print("=" * 90)

print(
    "Project root :",
    PROJECT_ROOT
)

print(
    "Train        :",
    TRAIN_PATH
)

print(
    "Validation   :",
    VALIDATION_PATH
)

print(
    "Test         :",
    TEST_PATH
)

print(
    "Output       :",
    OUTPUT_DIR
)


train = pd.read_csv(
    TRAIN_PATH
)

validation = pd.read_csv(
    VALIDATION_PATH
)

test = pd.read_csv(
    TEST_PATH
)


print("\n" + "=" * 90)
print("DATASET SHAPES")
print("=" * 90)

print(
    "Train      :",
    train.shape
)

print(
    "Validation :",
    validation.shape
)

print(
    "Test       :",
    test.shape
)


TARGET = "label"

assert TARGET in train.columns
assert TARGET in validation.columns
assert TARGET in test.columns

# Same schema across all three splits
assert (
    list(train.columns)
    ==
    list(validation.columns)
)

assert (
    list(train.columns)
    ==
    list(test.columns)
)


print("\nLABEL DISTRIBUTION")

print("\nTRAIN")
print(
    train[TARGET]
    .value_counts()
    .sort_index()
)

print("\nVALIDATION")
print(
    validation[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTEST")
print(
    test[TARGET]
    .value_counts()
    .sort_index()
)

RANDOM_STATE = 42

FORBIDDEN_METADATA = {
    "label",
    "pair_id",
    "source_id",
    "target_id",
    "bank_id",
    "settlement_id",
    "ground_truth",
    "truth",
    "match_id",
    "event_id",
    "event_link",
    "event_links",
}


REFERENCE_FEATURES = [
    "reference_exact_match",
    "utr_similarity",
    "settlement_id_in_description",
    "description_token_overlap",
]


print("\n" + "=" * 90)
print("BASIC DATA QUALITY AUDIT")
print("=" * 90)

print(
    "Train missing cells:",
    int(
        train.isna()
        .sum()
        .sum()
    )
)

print(
    "Validation missing cells:",
    int(
        validation.isna()
        .sum()
        .sum()
    )
)

print(
    "Test missing cells:",
    int(
        test.isna()
        .sum()
        .sum()
    )
)

print(
    "\nTrain duplicate rows:",
    int(
        train.duplicated()
        .sum()
    )
)

print(
    "Validation duplicate rows:",
    int(
        validation.duplicated()
        .sum()
    )
)

print(
    "Test duplicate rows:",
    int(
        test.duplicated()
        .sum()
    )
)


forbidden_present = sorted(
    [
        c
        for c in train.columns
        if c in FORBIDDEN_METADATA
        and c != TARGET
    ]
)

print(
    "\nForbidden predictor metadata:",
    forbidden_present
)

if forbidden_present:

    raise RuntimeError(
        "STOP: forbidden metadata columns detected."
    )

print(
    "Metadata check: PASSED"
)


print("\n" + "=" * 90)
print("GROUPED SOURCE-ID LEAKAGE AUDIT")
print("=" * 90)

if "source_id" not in train.columns:

    print(
        "STATUS: NOT VERIFIABLE"
    )

    print(
        "Reason: source_id was intentionally removed from the "
        "final v3 train/validation/test CSVs."
    )

    print(
        "The v3 generation pipeline performs grouped splitting "
        "before identifier removal."
    )

    SOURCE_AUDIT_STATUS = (
        "NOT_VERIFIABLE"
    )

else:

    train_sources = set(
        train["source_id"]
        .dropna()
        .astype(str)
    )

    val_sources = set(
        validation["source_id"]
        .dropna()
        .astype(str)
    )

    test_sources = set(
        test["source_id"]
        .dropna()
        .astype(str)
    )

    tv = (
        train_sources
        &
        val_sources
    )

    tt = (
        train_sources
        &
        test_sources
    )

    vt = (
        val_sources
        &
        test_sources
    )

    print(
        "Train ↔ Validation:",
        len(tv)
    )

    print(
        "Train ↔ Test:",
        len(tt)
    )

    print(
        "Validation ↔ Test:",
        len(vt)
    )

    if tv or tt or vt:

        raise RuntimeError(
            "STOP: source_id leakage detected."
        )

    SOURCE_AUDIT_STATUS = "PASSED"

    print(
        "source_id grouped isolation: PASSED"
    )


def make_feature_groups(
    df: pd.DataFrame
):

    all_features = [
        c
        for c in df.columns
        if c not in FORBIDDEN_METADATA
    ]


    amount_features = [
        c
        for c in [
            "bank_amount",
            "settlement_net_amount",
            "settlement_gross_amount",
            "settlement_fee_amount",
            "settlement_fee_tax_amount",
            "settlement_refund_amount",
            "amount_difference",
            "absolute_amount_difference",
            "relative_amount_difference",
            "amount_ratio",
            "exact_amount_match",
            "absolute_amount_difference_vs_gross",
        ]
        if c in df.columns
    ]


    date_features = [
        c
        for c in [
            "date_difference_days",
            "value_date_difference_days",
            "same_day",
            "within_1_day",
            "within_3_days",
            "within_7_days",
        ]
        if c in df.columns
    ]


    reference_features = [
        c
        for c in REFERENCE_FEATURES
        if c in df.columns
    ]


    entity_features = [
        c
        for c in [
            "merchant_match"
        ]
        if c in df.columns
    ]


    context_features = [
        c
        for c in [
            "transaction_type_is_credit",
            "bank_category",
            "payment_count",
        ]
        if c in df.columns
    ]


    ambiguity_features = [
        c
        for c in [
            "candidate_count",
            "candidate_rank_by_amount",
            "candidate_rank_by_date",
            "candidate_rank_by_reference",
            "is_best_amount_candidate",
            "is_best_date_candidate",
            "is_best_reference_candidate",
            "best_vs_second_best_amount_gap",
            "best_vs_second_best_date_gap",
            "best_vs_second_best_reference_gap",
        ]
        if c in df.columns
    ]


    no_amount = [
        c
        for c in all_features
        if c not in set(
            amount_features
        )
    ]


    no_amount_no_date = [
        c
        for c in all_features
        if c not in (
            set(amount_features)
            |
            set(date_features)
        )
    ]


    return {

        "ALL_FEATURES":
            all_features,

        "AMOUNT_ONLY":
            amount_features,

        "DATE_ONLY":
            date_features,

        "REFERENCE_TEXT_ONLY":
            reference_features,

        "ENTITY_ONLY":
            entity_features,

        "CONTEXT_ONLY":
            context_features,

        "NO_AMOUNT":
            no_amount,

        "NO_AMOUNT_NO_DATE":
            no_amount_no_date,

        "AMBIGUITY_ONLY":
            ambiguity_features,
    }


feature_groups = (
    make_feature_groups(train)
)


print("\n" + "=" * 90)
print("FEATURE GROUPS")
print("=" * 90)

for group_name, features in (
    feature_groups.items()
):

    print(
        f"\n{group_name} "
        f"({len(features)} features)"
    )

    print(
        features
    )


for group_name, features in (
    feature_groups.items()
):

    missing = [
        c
        for c in features
        if c not in train.columns
    ]

    if missing:

        raise RuntimeError(
            f"{group_name} missing features: {missing}"
        )

print(
    "\nFEATURE GROUP VALIDATION: PASSED"
)


REFERENCE_FEATURES_INFERENCE_SAFE = True

print("\n" + "=" * 90)
print("REFERENCE / TEXT PROVENANCE")
print("=" * 90)

print(
    "reference_exact_match : observed bank ↔ candidate settlement"
)

print(
    "utr_similarity        : observed bank ↔ candidate settlement"
)

print(
    "description features  : observed bank ↔ candidate settlement"
)

print(
    "candidate ranking     : candidate-set based"
)

print(
    "Ground-truth-derived predictor features: NONE"
)

print(
    "\nREFERENCE/TEXT PROVENANCE VERIFIED."
)


def run_reference_audit(
    train_df,
    validation_df,
    test_df
):

    rows = []

    for split_name, df in [
        ("train", train_df),
        ("validation", validation_df),
        ("test", test_df)
    ]:

        y = (
            df[TARGET]
            .astype(int)
        )

        for feature in REFERENCE_FEATURES:

            if feature not in df.columns:
                continue

            x = df[feature]

            row = {

                "split":
                    split_name,

                "feature":
                    feature,

                "dtype":
                    str(x.dtype),

                "nunique":
                    int(
                        x.nunique(
                            dropna=False
                        )
                    )
            }

            if pd.api.types.is_numeric_dtype(
                x
            ):

                values = (
                    pd.to_numeric(
                        x,
                        errors="coerce"
                    )
                    .fillna(0)
                )

                try:

                    auc = (
                        roc_auc_score(
                            y,
                            values
                        )
                    )

                except Exception:

                    auc = np.nan

                row[
                    "roc_auc"
                ] = float(auc)

                truthy = (
                    values != 0
                )

                if truthy.any():

                    row[
                        "positive_rate_when_truthy"
                    ] = float(
                        y[truthy].mean()
                    )

                else:

                    row[
                        "positive_rate_when_truthy"
                    ] = np.nan

            else:

                encoded = (
                    pd.factorize(
                        x.fillna(
                            "<NA>"
                        )
                        .astype(str)
                    )[0]
                )

                try:

                    auc = (
                        roc_auc_score(
                            y,
                            encoded
                        )
                    )

                except Exception:

                    auc = np.nan

                row[
                    "roc_auc"
                ] = float(auc)

                grouped = pd.crosstab(
                    x.fillna(
                        "<NA>"
                    )
                    .astype(str),
                    y,
                    normalize="index"
                )

                if not grouped.empty:

                    row[
                        "max_group_purity"
                    ] = float(
                        grouped
                        .max(axis=1)
                        .max()
                    )

                else:

                    row[
                        "max_group_purity"
                    ] = np.nan

            rows.append(
                row
            )

    return pd.DataFrame(
        rows
    )


reference_audit = (
    run_reference_audit(
        train,
        validation,
        test
    )
)

print(
    reference_audit
    .to_string(index=False)
)

reference_audit.to_csv(
    OUTPUT_DIR /
    "reference_text_audit.csv",
    index=False
)


print("\n" + "=" * 90)
print("DIRECT exact_amount_match DIAGNOSTIC")
print("=" * 90)

if "exact_amount_match" in train.columns:

    y = (
        train[TARGET]
        .astype(int)
    )

    exact_amount = (
        train[
            "exact_amount_match"
        ]
        .fillna(0)
        .astype(int)
    )

    accuracy = (
        exact_amount
        ==
        y
    ).mean()

    print(
        "Training accuracy using exact_amount_match alone:",
        round(
            float(accuracy),
            6
        )
    )

    print(
        "\nExact amount match distribution by class:"
    )

    print(
        pd.crosstab(
            y,
            exact_amount,
            normalize="index"
        )
    )


def make_preprocessor(
    X: pd.DataFrame
):

    numeric_columns = (
        X
        .select_dtypes(
            include=[
                np.number,
                "bool"
            ]
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        c
        for c in X.columns
        if c not in numeric_columns
    ]

    transformers = []

    if numeric_columns:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            )
        ])

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns
            )
        )


    if categorical_columns:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),

            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )


    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


def build_models(
    X
):

    estimators = {

        "Logistic Regression":
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=500,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
    }


    if XGBClassifier is not None:

        estimators[
            "XGBoost"
        ] = XGBClassifier(

            n_estimators=400,

            max_depth=5,

            learning_rate=0.05,

            subsample=0.90,

            colsample_bytree=0.90,

            objective="binary:logistic",

            eval_metric="logloss",

            random_state=RANDOM_STATE,

            n_jobs=-1,

            tree_method="hist"
        )


    if LGBMClassifier is not None:

        estimators[
            "LightGBM"
        ] = LGBMClassifier(

            n_estimators=400,

            num_leaves=31,

            learning_rate=0.05,

            subsample=0.90,

            colsample_bytree=0.90,

            random_state=RANDOM_STATE,

            verbosity=-1,

            n_jobs=-1
        )


    pipelines = {}

    for name, estimator in (
        estimators.items()
    ):

        pipelines[name] = Pipeline([
            (
                "preprocess",
                make_preprocessor(X)
            ),

            (
                "model",
                estimator
            )
        ])

    return pipelines


def balanced_sample_weights(
    y
):

    y = (
        np.asarray(y)
        .astype(int)
    )

    n = len(y)

    positive_count = max(
        int(y.sum()),
        1
    )

    negative_count = max(
        n - positive_count,
        1
    )

    positive_weight = (
        n /
        (2 * positive_count)
    )

    negative_weight = (
        n /
        (2 * negative_count)
    )

    return np.where(
        y == 1,
        positive_weight,
        negative_weight
    )


def predict_probability(
    model,
    X
):

    if hasattr(
        model,
        "predict_proba"
    ):

        return (
            model
            .predict_proba(X)[:, 1]
        )


    if hasattr(
        model,
        "decision_function"
    ):

        decision = (
            model
            .decision_function(X)
        )

        decision = np.clip(
            decision,
            -50,
            50
        )

        return (
            1.0 /
            (
                1.0 +
                np.exp(-decision)
            )
        )


    raise RuntimeError(
        "Model does not provide probability output."
    )


def find_best_threshold(
    y_true,
    scores
):

    y_true = (
        np.asarray(y_true)
        .astype(int)
    )

    thresholds = np.linspace(
        0.01,
        0.99,
        197
    )

    best_threshold = 0.50
    best_macro_f1 = -1.0

    for threshold in thresholds:

        predictions = (
            scores >= threshold
        ).astype(int)

        current_macro_f1 = (
            f1_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0
            )
        )

        if (
            current_macro_f1
            >
            best_macro_f1
        ):

            best_macro_f1 = (
                current_macro_f1
            )

            best_threshold = (
                threshold
            )

    return (
        float(best_threshold),
        float(best_macro_f1)
    )


def evaluate_scores(
    y_true,
    scores,
    threshold
):

    y_true = (
        np.asarray(y_true)
        .astype(int)
    )

    predictions = (
        scores >= threshold
    ).astype(int)

    return {

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    predictions
                )
            ),

        "precision":
            float(
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ),

        "recall":
            float(
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ),

        "f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    average="binary",
                    zero_division=0
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    average="macro",
                    zero_division=0
                )
            ),

        "pr_auc":
            float(
                average_precision_score(
                    y_true,
                    scores
                )
            ),

        "roc_auc":
            float(
                roc_auc_score(
                    y_true,
                    scores
                )
            ),

        "threshold":
            float(threshold),

        "predicted_positive_rate":
            float(
                predictions.mean()
            ),

        "confusion_matrix":
            confusion_matrix(
                y_true,
                predictions
            ).tolist()
    }


ALL_FEATURES = (
    feature_groups[
        "ALL_FEATURES"
    ]
)

X_train = (
    train[
        ALL_FEATURES
    ].copy()
)

X_val = (
    validation[
        ALL_FEATURES
    ].copy()
)

y_train = (
    train[TARGET]
    .astype(int)
)

y_val = (
    validation[TARGET]
    .astype(int)
)


models = build_models(
    X_train
)

model_results = []
trained_models = {}

sample_weights = (
    balanced_sample_weights(
        y_train
    )
)


print("\n" + "=" * 100)
print("TRAINING FINAL MODEL CANDIDATES")
print("=" * 100)


for model_name, model in (
    models.items()
):

    print(
        f"\nTraining {model_name}..."
    )

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )

        model.fit(
            X_train,
            y_train,
            model__sample_weight=
                sample_weights
        )


    val_scores = (
        predict_probability(
            model,
            X_val
        )
    )


    best_threshold, _ = (
        find_best_threshold(
            y_val,
            val_scores
        )
    )


    metrics = (
        evaluate_scores(
            y_val,
            val_scores,
            best_threshold
        )
    )


    model_results.append({

        "model":
            model_name,

        "feature_count":
            len(ALL_FEATURES),

        "validation_accuracy":
            metrics["accuracy"],

        "validation_precision":
            metrics["precision"],

        "validation_recall":
            metrics["recall"],

        "validation_f1":
            metrics["f1"],

        "validation_macro_f1":
            metrics["macro_f1"],

        "validation_pr_auc":
            metrics["pr_auc"],

        "validation_roc_auc":
            metrics["roc_auc"],

        "validation_threshold":
            metrics["threshold"]
    })


    trained_models[
        model_name
    ] = model


    print(
        f"{model_name}: "
        f"Macro-F1={metrics['macro_f1']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"PR-AUC={metrics['pr_auc']:.4f} | "
        f"ROC-AUC={metrics['roc_auc']:.4f} | "
        f"Threshold={metrics['threshold']:.4f}"
    )


model_results_df = (
    pd.DataFrame(
        model_results
    )
    .sort_values(
        [
            "validation_macro_f1",
            "validation_pr_auc",
            "validation_roc_auc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n" +
    "=" * 100
)

print(
    "MODEL COMPARISON"
)

print(
    "=" * 100
)

print(
    model_results_df.to_string(
        index=False
    )
)


BEST_MODEL_NAME = (
    model_results_df
    .iloc[0]["model"]
)

BEST_THRESHOLD = float(
    model_results_df
    .iloc[0]["validation_threshold"]
)

BEST_MODEL = (
    trained_models[
        BEST_MODEL_NAME
    ]
)


print(
    "\n" +
    "=" * 100
)

print(
    "SELECTED MODEL"
)

print(
    "=" * 100
)

print(
    "Model     :",
    BEST_MODEL_NAME
)

print(
    "Threshold :",
    BEST_THRESHOLD
)

print(
    "Features  :",
    len(ALL_FEATURES)
)


ABLATION_GROUPS = {

    "AMOUNT_ONLY":
        feature_groups[
            "AMOUNT_ONLY"
        ],

    "DATE_ONLY":
        feature_groups[
            "DATE_ONLY"
        ],

    "REFERENCE_TEXT_ONLY":
        feature_groups[
            "REFERENCE_TEXT_ONLY"
        ],

    "ENTITY_ONLY":
        feature_groups[
            "ENTITY_ONLY"
        ],

    "CONTEXT_ONLY":
        feature_groups[
            "CONTEXT_ONLY"
        ],

    "NO_AMOUNT":
        feature_groups[
            "NO_AMOUNT"
        ],

    "NO_AMOUNT_NO_DATE":
        feature_groups[
            "NO_AMOUNT_NO_DATE"
        ],
}


ablation_rows = []


print(
    "\n" +
    "=" * 100
)

print(
    "ABLATION STUDY"
)

print(
    "=" * 100
)


for experiment_name, features in (
    ABLATION_GROUPS.items()
):

    if not features:
        continue

    X_tr = (
        train[
            features
        ].copy()
    )

    X_va = (
        validation[
            features
        ].copy()
    )

    for model_name, model in (
        build_models(X_tr).items()
    ):

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                X_tr,
                y_train,
                model__sample_weight=
                    sample_weights
            )

        scores = (
            predict_probability(
                model,
                X_va
            )
        )

        threshold, _ = (
            find_best_threshold(
                y_val,
                scores
            )
        )

        metrics = (
            evaluate_scores(
                y_val,
                scores,
                threshold
            )
        )

        ablation_rows.append({

            "experiment":
                experiment_name,

            "model":
                model_name,

            "feature_count":
                len(features),

            "validation_macro_f1":
                metrics["macro_f1"],

            "validation_f1":
                metrics["f1"],

            "validation_pr_auc":
                metrics["pr_auc"],

            "validation_roc_auc":
                metrics["roc_auc"],

            "threshold":
                threshold
        })


ablation_df = (
    pd.DataFrame(
        ablation_rows
    )
    .sort_values(
        [
            "experiment",
            "validation_macro_f1"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    ablation_df.to_string(
        index=False
    )
)


ablation_df.to_csv(
    OUTPUT_DIR /
    "ablation_results.csv",
    index=False
)


# =====================================================================
# CELL 23 — REFERENCE/TEXT SHORTCUT SUMMARY
# =====================================================================

print(
    "\n" +
    "=" * 100
)

print(
    "REFERENCE/TEXT SHORTCUT SUMMARY"
)

print(
    "=" * 100
)


reference_ablation = (
    ablation_df[
        ablation_df[
            "experiment"
        ]
        ==
        "REFERENCE_TEXT_ONLY"
    ]
)

no_amount_ablation = (
    ablation_df[
        ablation_df[
            "experiment"
        ]
        ==
        "NO_AMOUNT"
    ]
)

no_amount_no_date_ablation = (
    ablation_df[
        ablation_df[
            "experiment"
        ]
        ==
        "NO_AMOUNT_NO_DATE"
    ]
)


print(
    "\nREFERENCE_TEXT_ONLY"
)

print(
    reference_ablation.to_string(
        index=False
    )
)


print(
    "\nNO_AMOUNT"
)

print(
    no_amount_ablation.to_string(
        index=False
    )
)


print(
    "\nNO_AMOUNT_NO_DATE"
)

print(
    no_amount_no_date_ablation.to_string(
        index=False
    )
)


FINAL_MODEL = (
    build_models(
        X_train
    )[
        BEST_MODEL_NAME
    ]
)

with warnings.catch_warnings():

    warnings.simplefilter(
        "ignore"
    )

    FINAL_MODEL.fit(
        X_train,
        y_train,
        model__sample_weight=
            sample_weights
    )


validation_scores = (
    predict_probability(
        FINAL_MODEL,
        X_val
    )
)

FINAL_THRESHOLD, _ = (
    find_best_threshold(
        y_val,
        validation_scores
    )
)

validation_metrics = (
    evaluate_scores(
        y_val,
        validation_scores,
        FINAL_THRESHOLD
    )
)


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL VALIDATION RESULT"
)

print(
    "=" * 100
)

print(
    json.dumps(
        validation_metrics,
        indent=2
    )
)


X_test = (
    test[
        ALL_FEATURES
    ].copy()
)

y_test = (
    test[TARGET]
    .astype(int)
)

test_scores = (
    predict_probability(
        FINAL_MODEL,
        X_test
    )
)

test_predictions = (
    test_scores
    >= FINAL_THRESHOLD
).astype(int)


test_metrics = (
    evaluate_scores(
        y_test,
        test_scores,
        FINAL_THRESHOLD
    )
)


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL TEST RESULT"
)

print(
    "=" * 100
)

print(
    json.dumps(
        test_metrics,
        indent=2
    )
)


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL TEST CLASSIFICATION REPORT"
)

print(
    "=" * 100
)

print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0
    )
)


confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_predictions
    ),
    index=[
        "true_0",
        "true_1"
    ],
    columns=[
        "pred_0",
        "pred_1"
    ]
)


print(
    "\n" +
    "=" * 80
)

print(
    "CONFUSION MATRIX"
)

print(
    "=" * 80
)

print(
    confusion
)


test_predictions_df = (
    test.copy()
)

test_predictions_df[
    "predicted_probability"
] = test_scores

test_predictions_df[
    "predicted_label"
] = test_predictions

test_predictions_df[
    "prediction_correct"
] = (
    test_predictions_df[
        TARGET
    ]
    ==
    test_predictions_df[
        "predicted_label"
    ]
)

TEST_PREDICTIONS_PATH = (
    OUTPUT_DIR /
    "test_predictions.csv"
)

test_predictions_df.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False
)


classification_report_df = (
    pd.DataFrame(
        classification_report(
            y_test,
            test_predictions,
            output_dict=True,
            zero_division=0
        )
    )
    .T
)


CLASSIFICATION_REPORT_PATH = (
    OUTPUT_DIR /
    "classification_report.csv"
)

classification_report_df.to_csv(
    CLASSIFICATION_REPORT_PATH
)


CONFUSION_MATRIX_PATH = (
    OUTPUT_DIR /
    "confusion_matrix.csv"
)

confusion.to_csv(
    CONFUSION_MATRIX_PATH
)


MODEL_PATH = (
    OUTPUT_DIR /
    "best_reconciliation_model.joblib"
)

model_artifact = {

    "model":
        FINAL_MODEL,

    "model_name":
        BEST_MODEL_NAME,

    "feature_names":
        ALL_FEATURES,

    "threshold":
        float(FINAL_THRESHOLD),

    "target":
        TARGET,

    "random_state":
        RANDOM_STATE,

    "reference_features":
        REFERENCE_FEATURES,

    "reference_features_inference_safe":
        REFERENCE_FEATURES_INFERENCE_SAFE,

    "pipeline_role":
        "residual_ambiguous_candidate_matching",

    "upstream_architecture":
        "deterministic_reconciliation_first"
}


joblib.dump(
    model_artifact,
    MODEL_PATH
)


feature_schema = {

    "target":
        TARGET,

    "feature_count":
        len(ALL_FEATURES),

    "features":
        ALL_FEATURES,

    "reference_features":
        REFERENCE_FEATURES,

    "reference_provenance_verified":
        REFERENCE_FEATURES_INFERENCE_SAFE,

    "model":
        BEST_MODEL_NAME,

    "threshold":
        float(FINAL_THRESHOLD),

    "dataset_role":
        "residual_ambiguous_matching"
}


FEATURE_SCHEMA_PATH = (
    OUTPUT_DIR /
    "feature_schema.json"
)


with open(
    FEATURE_SCHEMA_PATH,
    "w"
) as f:

    json.dump(
        feature_schema,
        f,
        indent=2
    )


final_metrics = {

    "selected_model":
        BEST_MODEL_NAME,

    "threshold":
        float(FINAL_THRESHOLD),

    "feature_count":
        len(ALL_FEATURES),

    "validation":
        validation_metrics,

    "test":
        test_metrics,

    "dataset_shapes": {

        "train":
            list(train.shape),

        "validation":
            list(validation.shape),

        "test":
            list(test.shape)
    },

    "source_id_group_audit":
        {
            "status":
                SOURCE_AUDIT_STATUS
        },

    "reference_provenance_verified":
        REFERENCE_FEATURES_INFERENCE_SAFE
}


FINAL_METRICS_PATH = (
    OUTPUT_DIR /
    "final_metrics.json"
)


with open(
    FINAL_METRICS_PATH,
    "w"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=2
    )


MODEL_COMPARISON_PATH = (
    OUTPUT_DIR /
    "model_comparison.csv"
)

model_results_df.to_csv(
    MODEL_COMPARISON_PATH,
    index=False
)


training_summary = {

    "status":
        "READY_FOR_RECONCILIATION_INTEGRATION",

    "selected_model":
        BEST_MODEL_NAME,

    "threshold":
        float(FINAL_THRESHOLD),

    "feature_count":
        len(ALL_FEATURES),

    "validation_macro_f1":
        validation_metrics[
            "macro_f1"
        ],

    "validation_f1":
        validation_metrics[
            "f1"
        ],

    "validation_pr_auc":
        validation_metrics[
            "pr_auc"
        ],

    "validation_roc_auc":
        validation_metrics[
            "roc_auc"
        ],

    "test_macro_f1":
        test_metrics[
            "macro_f1"
        ],

    "test_f1":
        test_metrics[
            "f1"
        ],

    "test_pr_auc":
        test_metrics[
            "pr_auc"
        ],

    "test_roc_auc":
        test_metrics[
            "roc_auc"
        ],

    "model_path":
        str(
            MODEL_PATH
        ),

    "reference_provenance_gate":
        REFERENCE_FEATURES_INFERENCE_SAFE,

    "source_id_group_audit":
        SOURCE_AUDIT_STATUS,

    "architecture_role":
        (
            "ML matching only after "
            "deterministic reconciliation"
        )
}


TRAINING_SUMMARY_PATH = (
    OUTPUT_DIR /
    "training_summary.json"
)


with open(
    TRAINING_SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        training_summary,
        f,
        indent=2
    )


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL ARTIFACT CHECK"
)

print(
    "=" * 100
)


artifact_checks = {

    "best_reconciliation_model.joblib":
        MODEL_PATH.exists(),

    "feature_schema.json":
        FEATURE_SCHEMA_PATH.exists(),

    "final_metrics.json":
        FINAL_METRICS_PATH.exists(),

    "training_summary.json":
        TRAINING_SUMMARY_PATH.exists(),

    "model_comparison.csv":
        MODEL_COMPARISON_PATH.exists(),

    "ablation_results.csv":
        (
            OUTPUT_DIR /
            "ablation_results.csv"
        ).exists(),

    "reference_text_audit.csv":
        (
            OUTPUT_DIR /
            "reference_text_audit.csv"
        ).exists(),

    "test_predictions.csv":
        TEST_PREDICTIONS_PATH.exists(),

    "classification_report.csv":
        CLASSIFICATION_REPORT_PATH.exists(),

    "confusion_matrix.csv":
        CONFUSION_MATRIX_PATH.exists(),
}


for name, exists in (
    artifact_checks.items()
):

    print(
        f"{name:45s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )


if not all(
    artifact_checks.values()
):

    raise RuntimeError(
        "One or more required artifacts were not created."
    )


print(
    "\n"
)

print(
    "=" * 110
)

print(
    "RECONCILIATION MODEL TRAINING COMPLETE"
)

print(
    "=" * 110
)

print(
    f"Selected model       : {BEST_MODEL_NAME}"
)

print(
    f"Final threshold      : {FINAL_THRESHOLD:.4f}"
)

print(
    f"Feature count        : {len(ALL_FEATURES)}"
)

print(
    f"Validation Macro-F1  : "
    f"{validation_metrics['macro_f1']:.4f}"
)

print(
    f"Validation F1        : "
    f"{validation_metrics['f1']:.4f}"
)

print(
    f"Validation PR-AUC    : "
    f"{validation_metrics['pr_auc']:.4f}"
)

print(
    f"Validation ROC-AUC   : "
    f"{validation_metrics['roc_auc']:.4f}"
)

print(
    f"Test Macro-F1        : "
    f"{test_metrics['macro_f1']:.4f}"
)

print(
    f"Test F1              : "
    f"{test_metrics['f1']:.4f}"
)

print(
    f"Test PR-AUC          : "
    f"{test_metrics['pr_auc']:.4f}"
)

print(
    f"Test ROC-AUC         : "
    f"{test_metrics['roc_auc']:.4f}"
)

print(
    f"Source-ID audit      : "
    f"{SOURCE_AUDIT_STATUS}"
)

print(
    f"Reference provenance : "
    f"{'VERIFIED' if REFERENCE_FEATURES_INFERENCE_SAFE else 'NOT VERIFIED'}"
)

print(
    "\nModel:"
)

print(
    MODEL_PATH
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\nSTATUS:"
)

print(
    "READY_FOR_RECONCILIATION_INTEGRATION"
)

print(
    "=" * 110
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IMPORTS COMPLETE
Pandas : 2.2.3
NumPy  : 2.1.3
XGBoost: available
LightGBM: available

PATH CONFIGURATION
Project root : /content/drive/MyDrive/Razorpay_AI
Train        : /content/drive/MyDrive/Razorpay_AI/ml_data/matching/train.csv
Validation   : /content/drive/MyDrive/Razorpay_AI/ml_data/matching/validation.csv
Test         : /content/drive/MyDrive/Razorpay_AI/ml_data/matching/test.csv
Output       : /content/drive/MyDrive/Razorpay_AI/ml/models/reconciliation

DATASET SHAPES
Train      : (1351, 36)
Validation : (289, 36)
Test       : (291, 36)

LABEL DISTRIBUTION

TRAIN
label
0    1272
1      79
Name: count, dtype: int64

VALIDATION
label
0    270
1     19
Name: count, dtype: int64

TEST
label
0    273
1     18
Name: count, dtype: int64

BASIC DATA QUALITY AUDIT
Train missing cells: 396
Validation missing cells: 75
Test missing cells: 96

Train duplicate ro